# The null controls, and what survives them

* **White noise**: what wavelet energies of pure noise look like.
* **Detrended random walk**: rolling detrending shapes multiscale energy by itself.
* **Shuffled returns**: the real return distribution, no serial dependence.
* **Block bootstrap** (64, 256, 1024-bar blocks): short-range dependence and volatility clustering inside blocks, nothing longer. Blocks from different eras also create volatility jumps at their joins - read this null for what it keeps.

The research ledger records every hypothesis tested, including the ones that failed.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features.wavelet*` and `xauusd_quant.research.wavelet_*` and read back from `results/wavelet_research/<tf>/<series>/window_<N>/`. Generate those first:

```
xq wavelet-research --timeframe 5m --source regression_residual --window 512
xq wavelet-research --timeframe 5m --all-windows
xq wavelet-research --all
xq wavelet-research --controls
```

**Causal vs offline.** Per-bar features are *causal*: a one-sided MODWT on the trailing window, identical whether or not later bars exist, and the only thing ever written to `data/features/wavelet/`. Scalograms and ridges are computed from whole slices, centred on every point, and are labelled **NON-CAUSAL - VISUALIZATION ONLY**; they never become features.

Every number sits beside the null controls: white noise, a detrended random walk (the same regression and OU pipeline on a series with no structure), shuffled returns and block-bootstrapped returns (blocks of 64, 256 and 1024 bars). No trading rule is defined or evaluated anywhere. No costs are applied. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.wavelet_config import load_wavelet_config

wavelet = load_wavelet_config(ROOT / "config" / "wavelet.yaml")

TIMEFRAME = "5m"                    # change and re-run
SERIES = "regression_residual"      # regression_residual | ou_innovation | log_return
WINDOW = 512                        # rolling window N (bars)

results = wavelet.results_dir(TIMEFRAME, SERIES, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} {SERIES} N={WINDOW}. Run:  xq wavelet-research "
        f"--timeframe {TIMEFRAME} --source {SERIES} --window {WINDOW}")

def table(name):
    """Load a previously generated wavelet table (None if it was not written)."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name, folder=None):
    path = (folder or results / "plots") / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

payload = json.loads((results / "summary.json").read_text(encoding="utf-8"))
summary = payload["summary"]
lineage = payload["provenance"]["dataset"]
print(f"bars       : {summary['observations']:,} ({summary['valid_feature_fraction']:.1%} "
      f"with a valid {WINDOW}-bar window)")
print(f"ticks      : {lineage['tick_dataset_version']} ({lineage['tick_dataset_status']})")
print(f"bars       : {lineage.get('bar_dataset_version')}")
print(f"code       : {payload['provenance']['code_fingerprint']}, "
      f"PyWavelets {payload['provenance']['packages'].get('PyWavelets')}")
if lineage.get("label"):
    print(f"WARNING    : {lineage['label']}")
for warning in payload.get("warnings", []):
    print(f"WARNING    : {warning}")

In [ ]:
table('null_controls')

In [ ]:
plot('null_comparison')

## Every hypothesis and its verdict (registered before any result)

In [ ]:
registered = json.loads((wavelet.results_path / "hypotheses_registered.json").read_text())
print("registered:", registered["registered_utc"])
ledger = pl.read_parquet(wavelet.ledger_path)
(ledger.filter(pl.col("hypothesis_id").str.starts_with("WAVE-") & (pl.col("timeframe") == TIMEFRAME)
               & (pl.col("input_series") == SERIES) & (pl.col("window") == WINDOW))
 .group_by("hypothesis_id", "verdict").len().sort("hypothesis_id"))

## Across every study: test counts and the IC chance rate

In [ ]:
counts = wavelet.results_path / "hypothesis_test_counts.csv"
pl.read_csv(counts) if counts.exists() else "run xq wavelet-research --compare"

In [ ]:
rates = wavelet.results_path / "ic_chance_rates.csv"
(pl.read_csv(rates).group_by("input_series", "source")
 .agg(pl.col("tests").sum(), pl.col("exceedances").sum(),
      pl.col("exceedances_stable_sign").sum())
 .sort("input_series", "source")) if rates.exists() else "run xq wavelet-research --compare"